# Homework: Build and Benchmark an Autocomplete System

## Context

Imagine you are building the search box for a recipe app. When a user types a prefix such as `"pa"`, the app should quickly return matching recipe names such as `"pancakes"` and `"pasta"`.

In this assignment you will implement two versions of an autocomplete system:

1. `TrieAutocomplete` — a prefix tree (trie)
2. `ListAutocomplete` — a simple baseline backed by a plain Python list

Then you will compare their performance on the same workload.

## Learning goals

- Practice implementing a tree-based data structure
- See why specialised data structures can outperform naïve baselines
- Write simple correctness tests
- Run a small benchmark and interpret the result


## Rules and deliverables

### What you must submit

A completed notebook containing:

- a working `TrieAutocomplete` (your code in Part A)
- a working `ListAutocomplete` (your code in Part B)
- passing public tests (Part C)
- benchmark results (Part E)
- scaling experiment results and answers (Part F)
- short written answers (Part G)

### Exact method names

Use these exact class and method names so the notebook tests and the instructor tests can run:

- `TrieNode`
- `TrieAutocomplete`
- `ListAutocomplete`
- `insert(self, word)`
- `contains(self, word)`
- `suggest(self, prefix)`

### Scope and assumptions

To keep the assignment focused, assume the input words:

- are strings
- are already lowercase
- contain only letters `a`–`z`

You do **not** need to handle spaces, punctuation, or uppercase conversion inside the classes.

### Allowed and disallowed tools

You may use Python lists, Python dictionaries, loops, recursion, and helper methods.  
You may **not** use any external autocomplete or trie library, or any built-in tool that already solves the assignment.

Follow your course policy on outside help.

### Additional grading note

The public tests below are only part of the grading. The instructor may also run additional tests on other inputs.


## Required behavior

### `insert(word)`
- Adds `word` to the data structure.
- If `word` is already present, do nothing (store **unique** words only).

### `contains(word)`
- Returns `True` if the exact word is stored, `False` otherwise.

### `suggest(prefix)`
- Returns a Python list of **all** stored words that start with `prefix`.
- The list must be in **lexicographic order**.
- If there are no matches, return `[]`.
- If `prefix == ""`, return **all** stored words in lexicographic order.

### Baseline requirement
`ListAutocomplete` must use a plain Python list and answer prefix queries by scanning the list linearly.

### Trie requirement
`TrieAutocomplete` must use a trie. A dictionary for children is allowed.


In [ ]:
# Imports — do not change
import random
import string
import time
from statistics import mean, stdev


## Small sample data for debugging

Use `SAMPLE_WORDS` while building your solution before moving to the larger benchmark dataset.


In [ ]:
SAMPLE_WORDS = [
    "applepie", "applesauce", "apricot", "banana", "blackberry",
    "blueberry", "brownie", "carrotcake", "cheesecake", "chili",
    "chocolate", "cookies", "curry", "dumplings", "falafel",
    "focaccia", "granola", "guacamole", "lasagna", "lemonade",
    "muffin", "omelette", "pancakes", "pasta", "pickles",
    "pizza", "pretzel", "quiche", "ramen", "risotto",
    "salad", "sandwich", "smoothie", "soup", "tacos",
    "waffles", "yogurt",
]

EXPECTED_PA = ["pancakes", "pasta"]
EXPECTED_CH = ["cheesecake", "chili", "chocolate"]


## Part A: Implement the trie

### Suggested design

Each `TrieNode` should store:
- `children` — a dict mapping a character to its child `TrieNode`
- `is_word` — `True` when a complete word ends at this node

For `suggest(prefix)`:
1. Walk down the trie along the prefix characters.
2. If the prefix is not found, return `[]`.
3. Otherwise do a depth-first search (DFS) from that node.
4. Collect all complete words found below it.
5. Return them in lexicographic order.

A simple way to guarantee lexicographic order in the DFS is:
```python
for ch in sorted(node.children):
    ...
```


In [ ]:
class TrieNode:
    """One node in the trie.

    Attributes
    ----------
    children : dict
        Maps a single character to the corresponding child TrieNode.
    is_word : bool
        True when a complete word ends at this node.
    """

    def __init__(self):
        # TODO: initialise children and is_word
        raise NotImplementedError


class TrieAutocomplete:
    """Autocomplete structure backed by a prefix tree (trie)."""

    def __init__(self):
        # TODO: create the root TrieNode
        raise NotImplementedError

    def insert(self, word):
        """Add *word* to the trie. If already present, do nothing."""
        # Hint: walk from self.root one character at a time,
        #       creating new TrieNode children as needed,
        #       then mark the final node as a word.
        raise NotImplementedError

    def contains(self, word):
        """Return True if *word* was inserted, False otherwise."""
        # Hint: walk from self.root; return False as soon as a
        #       character is missing; check is_word at the end.
        raise NotImplementedError

    def suggest(self, prefix):
        """Return a sorted list of all stored words that start with *prefix*.

        Return [] if no matches exist.
        If prefix == '', return all stored words in lexicographic order.
        """
        # Step 1: walk down the trie to the node matching the last
        #         character of prefix (return [] if not found).
        # Step 2: collect all complete words below that node using a
        #         depth-first search.  Iterate sorted(node.children)
        #         to get lexicographic order for free.
        raise NotImplementedError

    # You may add private helper methods below, e.g. a _dfs method.


## Part B: Implement the baseline


In [ ]:
class ListAutocomplete:
    """Autocomplete structure backed by a plain Python list.

    All three operations must scan the list linearly — this is the
    intentional O(n) baseline.
    """

    def __init__(self):
        # TODO: initialise an empty list to store words
        raise NotImplementedError

    def insert(self, word):
        """Append *word* only if it is not already present."""
        # Hint: check membership before appending.
        raise NotImplementedError

    def contains(self, word):
        """Return True if *word* is in the list, False otherwise."""
        raise NotImplementedError

    def suggest(self, prefix):
        """Return a sorted list of all stored words that start with *prefix*."""
        # Hint: use str.startswith and sorted().
        raise NotImplementedError


## Part C: Public tests

Run these after you have implemented both classes. They are intentionally small and readable — their purpose is to make the expected behaviour precise.


In [ ]:
def build_structure(cls, words):
    """Helper: create an instance of *cls* and insert all *words*."""
    s = cls()
    for word in words:
        s.insert(word)
    return s


def run_public_tests():
    # ── Test 1: empty structures ──────────────────────────────────────────
    trie = TrieAutocomplete()
    lst  = ListAutocomplete()

    assert trie.contains("pizza") is False
    assert lst.contains("pizza")  is False
    assert trie.suggest("pi") == []
    assert lst.suggest("pi")  == []
    assert trie.suggest("") == []
    assert lst.suggest("") == []

    # ── Test 2: insert sample words ───────────────────────────────────────
    trie = build_structure(TrieAutocomplete, SAMPLE_WORDS)
    lst  = build_structure(ListAutocomplete,  SAMPLE_WORDS)

    # ── Test 3: exact membership ──────────────────────────────────────────
    assert trie.contains("pizza") is True
    assert lst.contains("pizza")  is True
    assert trie.contains("piz") is False   # prefix only — not a full word
    assert lst.contains("piz")  is False

    # ── Test 4: prefix suggestions ────────────────────────────────────────
    assert trie.suggest("pa") == EXPECTED_PA
    assert lst.suggest("pa")  == EXPECTED_PA
    assert trie.suggest("ch") == EXPECTED_CH
    assert lst.suggest("ch")  == EXPECTED_CH

    # ── Test 5: no matches ────────────────────────────────────────────────
    assert trie.suggest("zzz") == []
    assert lst.suggest("zzz")  == []

    # ── Test 6: empty prefix returns everything in lexicographic order ────
    expected_all = sorted(SAMPLE_WORDS)
    assert trie.suggest("") == expected_all
    assert lst.suggest("") == expected_all

    # ── Test 7: duplicate inserts must not create duplicates ──────────────
    trie.insert("pizza")
    trie.insert("pizza")
    lst.insert("pizza")
    lst.insert("pizza")
    assert trie.suggest("pizza").count("pizza") == 1
    assert lst.suggest("pizza").count("pizza")  == 1

    # ── Test 8: both implementations must agree on several prefixes ───────
    for prefix in ["a", "ap", "b", "bl", "c", "ca", "p", "pi", "s", ""]:
        assert trie.suggest(prefix) == lst.suggest(prefix), (
            f"Mismatch on prefix {prefix!r}: "
            f"trie={trie.suggest(prefix)}, list={lst.suggest(prefix)}"
        )

    print("All public tests passed.")


# Uncomment after implementing both classes:
# run_public_tests()


## Part D: Benchmark setup

**Do not change the parameters below.** Everyone should measure the same workload.

### Parameters (2× the original assignment)
- `NUM_EXPERIMENTS = 5`
- `NUM_INITIAL_INSERTS = 8000`
- `NUM_ADDITIONAL_INSERTS = 4000`
- `NUM_PREFIX_QUERIES = 6000`
- `NUM_CONTAINS_QUERIES = 6000`


In [ ]:
def make_word_dataset(n_words=24000, seed=42):
    """Generate a synthetic vocabulary of *n_words* unique lowercase words."""
    rng = random.Random(seed)
    stems = [
        "ap", "ba", "be", "bl", "br", "ca", "ch", "cl", "co", "cr",
        "fa", "fo", "fr", "gl", "gr", "la", "le", "ma", "mo", "pa",
        "pe", "pi", "po", "ra", "re", "sa", "sm", "so", "st", "wa",
    ]
    syllables = [
        "na", "ne", "ni", "no", "ra", "re", "ri", "ro", "la", "le",
        "li", "lo", "ta", "te", "ti", "to", "ma", "me", "mi", "mo",
        "ca", "ce", "ci", "co", "fa", "fe", "fi", "fo",
    ]
    words = set()
    while len(words) < n_words:
        stem = rng.choice(stems)
        suffix = "".join(rng.choice(syllables) for _ in range(rng.randint(2, 4)))
        words.add(stem + suffix)
    return sorted(words)


def make_contains_queries(words, n_queries, seed=123):
    rng = random.Random(seed)
    present = [rng.choice(words) for _ in range(n_queries // 2)]
    absent, letters, word_set = [], string.ascii_lowercase, set(words)
    while len(absent) < n_queries - len(present):
        base = list(rng.choice(words))
        idx = rng.randrange(len(base))
        original = base[idx]
        base[idx] = rng.choice([c for c in letters if c != original])
        candidate = "".join(base)
        if candidate not in word_set:
            absent.append(candidate)
    queries = present + absent
    rng.shuffle(queries)
    return queries


def make_prefix_queries(words, n_queries, seed=456):
    rng = random.Random(seed)
    return [
        w[: rng.randint(1, min(4, len(w)))]
        for w in (rng.choice(words) for _ in range(n_queries))
    ]


NUM_EXPERIMENTS        = 5
NUM_INITIAL_INSERTS    = 8000
NUM_ADDITIONAL_INSERTS = 4000
NUM_PREFIX_QUERIES     = 6000
NUM_CONTAINS_QUERIES   = 6000

DATASET          = make_word_dataset()
INITIAL_WORDS    = DATASET[:NUM_INITIAL_INSERTS]
ADDITIONAL_WORDS = DATASET[NUM_INITIAL_INSERTS : NUM_INITIAL_INSERTS + NUM_ADDITIONAL_INSERTS]
PREFIX_QUERIES   = make_prefix_queries(DATASET, NUM_PREFIX_QUERIES)
CONTAINS_QUERIES = make_contains_queries(DATASET, NUM_CONTAINS_QUERIES)


In [ ]:
def make_operations(additional_words, prefix_queries, contains_queries, seed=999):
    rng = random.Random(seed)
    operations = (
        [("insert",   w) for w in additional_words]
        + [("suggest",  p) for p in prefix_queries]
        + [("contains", w) for w in contains_queries]
    )
    rng.shuffle(operations)
    return operations


def benchmark_structure(structure_cls, operations):
    structure = structure_cls()
    start = time.perf_counter()
    for word in INITIAL_WORDS:
        structure.insert(word)
    for op, value in operations:
        if   op == "insert":   structure.insert(value)
        elif op == "suggest":  structure.suggest(value)
        elif op == "contains": structure.contains(value)
        else: raise ValueError(f"Unknown operation: {op}")
    return time.perf_counter() - start


def run_benchmark():
    trie_times, list_times = [], []
    for i in range(NUM_EXPERIMENTS):
        ops = make_operations(
            ADDITIONAL_WORDS, PREFIX_QUERIES, CONTAINS_QUERIES, seed=999 + i
        )
        trie_times.append(benchmark_structure(TrieAutocomplete, ops))
        list_times.append(benchmark_structure(ListAutocomplete,  ops))

    print("TrieAutocomplete times:", trie_times)
    print("ListAutocomplete times:", list_times)
    print()
    print("TrieAutocomplete mean:", mean(trie_times))
    print("ListAutocomplete mean:", mean(list_times))
    if len(trie_times) > 1:
        print("TrieAutocomplete std:", stdev(trie_times))
        print("ListAutocomplete std:", stdev(list_times))


## Part E: Run the benchmark

> **Your task:** Run the cell. Make sure the output appears in your submitted notebook.

Run the cell below **after** your public tests pass.


In [ ]:
run_benchmark()


## Part F: Scaling experiment

> **Your task:** Run all cells in this part, then answer the questions in the markdown cells below each experiment.

The Part E benchmark tells you which structure is faster on one fixed workload. This part asks you to measure *how that advantage changes* as the vocabulary grows, and to separate the two operations.

**Setup.** Both structures are pre-loaded before the timer starts, so insert cost does not affect the results. The number of queries is held constant across all sizes so you are isolating the effect of vocabulary size.

Fixed parameters:
- `SCALE_SIZES = [500, 1000, 2000, 4000, 8000]`
- `N_QUERIES = 300` (constant across all sizes)
- `N_REPEATS = 3` (mean of 3 runs reported)


In [ ]:
SCALE_SIZES   = [500, 1000, 2000, 4000, 8000]
N_QUERIES     = 300
N_REPEATS     = 3
SCALE_DATASET = make_word_dataset(n_words=10000, seed=42)


def build(cls, words):
    """Insert all words into a fresh structure and return it."""
    s = cls()
    for w in words:
        s.insert(w)
    return s


def time_queries(structure, queries, op):
    """Time *op* ('contains' or 'suggest') for all items in *queries*.
    Returns elapsed time in milliseconds."""
    fn = getattr(structure, op)
    t0 = time.perf_counter()
    for q in queries:
        fn(q)
    return (time.perf_counter() - t0) * 1000


def run_scaling_experiment(op, prefix_len_range=None):
    """Run the scaling experiment for a single operation.

    Parameters
    ----------
    op : 'contains' or 'suggest'
    prefix_len_range : (min_len, max_len) used when op == 'suggest'.
    """
    header = f"{'n_words':>8}  {'Trie (ms)':>10}  {'List (ms)':>10}  {'Speedup':>9}"
    print(header)
    print("-" * len(header))

    for n in SCALE_SIZES:
        words = SCALE_DATASET[:n]
        rng   = random.Random(77)

        if op == "contains":
            queries = [rng.choice(words) for _ in range(N_QUERIES)]
        else:
            lo, hi  = prefix_len_range
            queries = [
                w[: rng.randint(lo, min(hi, len(w)))]
                for w in (rng.choice(words) for _ in range(N_QUERIES))
            ]

        trie = build(TrieAutocomplete, words)
        lst  = build(ListAutocomplete,  words)

        t_trie = mean(time_queries(trie, queries, op) for _ in range(N_REPEATS))
        t_list = mean(time_queries(lst,  queries, op) for _ in range(N_REPEATS))

        speedup = t_list / t_trie if t_trie > 0 else float("inf")
        print(f"{n:>8}  {t_trie:>10.2f}  {t_list:>10.2f}  {speedup:>9.1f}x")


### F.1 — `contains` queries


In [ ]:
print("=== contains queries ===")
run_scaling_experiment("contains")


**Questions — answer in the cell below.**

1. As `n_words` doubles, roughly how does the **List** time change? Is that consistent with O(n) complexity?
2. As `n_words` doubles, roughly how does the **Trie** time change? What complexity does that suggest?
3. Does the speedup column grow, shrink, or stay roughly flat as `n_words` increases? Why?


**Your answers here.**


### F.2 — `suggest` queries with longer prefixes (length 3–4)

A longer prefix narrows the search to a small subset of the vocabulary.


In [ ]:
print("=== suggest queries (prefix length 3-4) ===")
run_scaling_experiment("suggest", prefix_len_range=(3, 4))


**Questions — answer in the cell below.**

1. Which structure is faster for longer prefixes, and by roughly how much at `n_words = 8000`?
2. How does the speedup change as `n_words` grows? Explain in terms of what each structure must do to answer the query.


**Your answers here.**


### F.3 — `suggest` queries with shorter prefixes (length 1–2)

A very short prefix like `"a"` or `"pa"` matches a large fraction of the vocabulary.


In [ ]:
print("=== suggest queries (prefix length 1-2) ===")
run_scaling_experiment("suggest", prefix_len_range=(1, 2))


**Questions — answer in the cell below.**

1. Which structure is faster here? Does the result surprise you?
2. The trie is supposed to be efficient by jumping directly to the matching subtree. Why is it *slower* than the list when the prefix is very short?
3. In a real-world autocomplete system only the top-5 or top-10 results are shown. How might limiting the number of returned results change this comparison?


**Your answers here.**


## Part G: Written answers

> **Your task:** Write your answers directly in this cell (replace the placeholder text).

1. Why is a trie a reasonable data structure for autocomplete?
2. Which implementation was faster in the Part E benchmark?
3. In 3–5 sentences, explain why you think the timing results came out that way.
4. Based on the scaling experiment, describe **one scenario** where you would prefer the trie and **one scenario** where you might prefer the list. Justify each choice.
5. Briefly describe one bug you encountered while implementing the assignment, and how you fixed it.


**Write your answers here.**


## Grading rubric

| Component | Weight |
|-----------|--------|
| Correctness (classes pass all tests) | 35 % |
| Data structure quality (genuine trie vs. genuine list) | 15 % |
| Benchmark (Part E run with results) | 15 % |
| Scaling experiment (Part F results + answers) | 20 % |
| Interpretation (Part G written answers) | 15 % |
